In [ ]:
import pandas as pd
import json

def extract_data():
    vendas = pd.read_csv('datasets/vendas_2023_2024.csv')
    produtos = pd.read_csv('datasets/produtos_raw.csv')

    with open('datasets/clientes_crm.json', 'r') as f:
        clientes_crm = pd.DataFrame(json.load(f))

    with open('datasets/custos_importacao.json', 'r') as f:
        custos_importacao = pd.DataFrame(json.load(f))

    return vendas, produtos, clientes_crm, custos_importacao

df_vendas, df_produtos, df_clientes_crm, df_custos_importacao = extract_data()

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/vendas_2023_2024.csv'

In [ ]:
def clean_products(df):
    # Remove o R$ e converte para float
    df['preco'] = df['preco'].str.replace('R\$', '').str.replace(',', '.').astype(float)

    # Dict de mapeamento para categorias
    category_mapping = {
        'E L E T R Ô N I C O S': 'Eletrônicos',
        'ELETRONICOS': 'Eletrônicos',
        'Eletrunicos': 'Eletrônicos',
        'eLeTrÔnIcOs': 'Eletrônicos',
        'eletrônicos': 'Eletrônicos',
        'Eletronicoz': 'Eletrônicos',
        'Eletroniscos': 'Eletrônicos',
        'eletronicos': 'Eletrônicos',
        'EletrônicoS': 'Eletrônicos',
        'P R O P U L S Ã O': 'Propulsão',
        'PROPULSÃO': 'Propulsão',
        'Prop': 'Propulsão',
        'Propulssão': 'Propulsão',
        'propulsao': 'Propulsão',
        'Propulção': 'Propulsão',
        'Propulssão': 'Propulsão',
        'PROPULSAO': 'Propulsão',
        'Propução': 'Propulsão',
        'pRoPuLsÃo': 'Propulsão',
        'Propulçao': 'Propulsão',
        'propulsão': 'Propulsão',
        'Propução': 'Propulsão',
        'PrOpUlSãO': 'Propulsão',
        'A N C O R A G E M': 'Ancoragem',
        'AnCoRaGeM': 'Ancoragem',
        'ANCORAGEM': 'Ancoragem',
        'Encoragem': 'Ancoragem',
        'Ancoraguem': 'Ancoragem',
        'Ancorajm': 'Ancoragem',
        'AncorageM': 'Ancoragem',
        'aNcOrAgEm': 'Ancoragem',
        'Ancorajem': 'Ancoragem',
        'Encoragi': 'Ancoragem',
        'Ancorajen': 'Ancoragem',
        'AncorajeM': 'Ancoragem',
        'Ancoragen': 'Ancoragem',
        'ancoragem': 'Ancoragem',

    }

    

In [6]:
import pandas as pd
import json
import re

# --- 1. CONFIGURAÇÕES E CARREGAMENTO ---
# Definimos a taxa de câmbio conforme premissa de negócio
TAXA_CAMBIO_BRL = 5.15 

def carregar_dados():
    vendas = pd.read_csv('datasets/vendas_2023_2024.csv')
    produtos = pd.read_csv('datasets/produtos_raw.csv')
    
    with open('datasets/clientes_crm.json', 'r', encoding='utf-8') as f:
        clientes = pd.DataFrame(json.load(f))
        
    with open('datasets/custos_importacao.json', 'r', encoding='utf-8') as f:
        custos_json = json.load(f)
        
    return vendas, produtos, clientes, custos_json

# --- 2. LIMPEZA DE PRODUTOS ---
def limpar_produtos(df):
    # Remove 'R$ ' e converte para float
    df['price'] = df['price'].str.replace('R$ ', '', regex=False).astype(float)
    
    # Dicionário de mapeamento para normalizar as categorias "sujas"
    mapa_categorias = {
        'ELETRONICOS': 'Eletrônicos', 'Eletrunicos': 'Eletrônicos', 'Eletronicoz': 'Eletrônicos',
        'PROPULSAO': 'Propulsão', 'Propulção': 'Propulsão', 'Prop': 'Propulsão',
        'Ancoragem': 'Ancoragem', 'Encoragem': 'Ancoragem', 'Ancorajm': 'Ancoragem'
        # ... outras variações identificadas na exploração
    }
    
    # Função auxiliar para lidar com espaços e acentos no mapeamento
    def padronizar(cat):
        cat_limpa = str(cat).upper().replace(' ', '')
        if 'ELETR' in cat_limpa: return 'Eletrônicos'
        if 'PROP' in cat_limpa: return 'Propulsão'
        if 'ANC' in cat_limpa or 'ENC' in cat_limpa: return 'Ancoragem'
        return 'Outros'

    df['category_clean'] = df['actual_category'].apply(padronizar)
    return df

# --- 3. LIMPEZA DE CLIENTES ---
def limpar_clientes(df):
    # Corrige o caractere '#' nos e-mails
    df['email'] = df['email'].str.replace('#', '@', regex=False)
    
    # Extrai a UF (Estado) usando Regex (procura por 2 letras maiúsculas isoladas)
    def extrair_uf(loc):
        match = re.search(r'\b([A-Z]{2})\b', loc)
        return match.group(1) if match else "N/A"
    
    df['state'] = df['location'].apply(extrair_uf)
    return df

# --- 4. TRATAMENTO DE CUSTOS (HISTÓRICO) ---
def processar_custos_historicos(custos_json):
    registros = []
    for item in custos_json:
        for hist in item['historic_data']:
            registros.append({
                'id_product': item['product_id'],
                'usd_price': hist['usd_price'],
                'data_inicio_custo': pd.to_datetime(hist['start_date'], dayfirst=True)
            })
    return pd.DataFrame(registros).sort_values('data_inicio_custo')

# --- 5. EXECUÇÃO DO ETL ---
vendas, produtos, clientes, custos_json = carregar_dados()

# Limpando bases secundárias
produtos = limpar_produtos(produtos)
clientes = limpar_clientes(clientes)
df_custos = processar_custos_historicos(custos_json)

# Padronizando datas de venda (suporta YYYY-MM-DD e DD-MM-YYYY)
vendas['sale_date'] = pd.to_datetime(vendas['sale_date'], errors='coerce').fillna(
    pd.to_datetime(vendas['sale_date'], format='%d-%m-%Y', errors='coerce')
)
vendas = vendas.dropna(subset=['sale_date']).sort_values('sale_date')

# Cruzamento Temporal: Busca o custo USD vigente na data da venda
vendas_master = pd.merge_asof(
    vendas, df_costs, 
    left_on='sale_date', right_on='data_inicio_custo', 
    by='id_product', direction='backward'
)

# Adicionando metadados de Produtos e Clientes
vendas_master = vendas_master.merge(produtos[['code', 'name', 'category_clean']], left_on='id_product', right_on='code', how='left')
vendas_master = vendas_master.merge(clientes[['code', 'full_name', 'state']], left_on='id_client', right_on='code', suffixes=('', '_client'), how='left')

# Cálculos Financeiros Finais
vendas_master['custo_total_brl'] = (vendas_master['usd_price'] * vendas_master['qtd'] * TAXA_CAMBIO_BRL).round(2)
vendas_master['lucro'] = (vendas_master['total'] - vendas_master['custo_total_brl']).round(2)

# Salvando resultado final
vendas_master.to_csv('lh_nautical_final_master.csv', index=False)

print("ETL Concluído com Sucesso!")

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/vendas_2023_2024.csv'